# Description

Notebook to create a sin based dataset

In [ ]:
import os
import sys

CURRENT_DIR = os.getcwd()
FOLDERS = CURRENT_DIR.split(os.sep)
TESIS_FOLDER_INDEX = FOLDERS.index('S-noise-gradient')
CURRENT_DIR = os.sep.join(FOLDERS[:TESIS_FOLDER_INDEX+1])
LIBS_PATH = os.path.join(CURRENT_DIR, 'src', 'libs')
assert os.path.exists(LIBS_PATH)
sys.path.append(LIBS_PATH)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import torch
import torch.nn as nn
import torch.optim as optim
from typing import List

# Locals
from utils import *
from dataset import Dataset
from models import *
import config as cfg
from dlnr import DLNoiseReduction

In [ ]:
DATA_PATH = os.path.join(CURRENT_DIR, 'data')
CHECKPOINT_PATH = os.path.join(CURRENT_DIR, 'checkpoints')
CONFIG_PATH = os.path.join(CURRENT_DIR, 'config')

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert device.type == "cuda"

In [ ]:
def polinomial_function_2variable(x0:float, x1:float):
    """
    Function that takes in two values x0 and x1 and returns its polinomial value.

    Args:
        x0 (float): 1st dimension value to be transformed.
        x1 (float): 2nd dimension value to be transformed.

    Returns:
        float: result of the polinomial function.
    """
    # return np.sin(np.sqrt(x0**2 + x1**2))
    return x0**2 + x1**2


def add_gaussian_noise(df:pd.DataFrame, columns:List[str], mean:float = 0.0, std:float = 0.1):
    """
    Adds Gaussian noise to specified columns in a DataFrame.

    Args:
        df (pandas.DataFrame): The DataFrame to which noise will be added.
        columns (list): A list of column names in the DataFrame to which noise will be added.
        mean (int, optional): The mean of the Gaussian distribution. Defaults to 0.
        std (float, optional): The standard deviation of the Gaussian distribution. Defaults to 0.1.

    Returns:
        pandas.DataFrame: The DataFrame with added Gaussian noise to specified columns.
    """
    for col in columns:
        df[col] = df[col] + np.random.normal(loc=mean, scale=std, size=len(df))
    return df

In [ ]:
X_train = np.array([np.random.normal(size=3000), np.random.normal(size=3000)]).T
X_train = pd.DataFrame(X_train, columns=['x0', 'x1'])
y_train = polinomial_function_2variable(X_train['x0'], X_train['x1'])

df_train = pd.concat([X_train, pd.DataFrame(y_train, columns=['y'])], axis=1)

In [ ]:
fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection='3d')

ax.scatter(df_train['x0'], df_train['x1'], df_train['y'], marker='o', c=df_train['y'], cmap='viridis', alpha=0.7)

ax.set_xlabel('x0')
ax.set_ylabel('x1')
ax.set_zlabel('y')

In [ ]:
df_val = df_train.copy()
df_test = df_train.copy()
df_noisy = df_train.copy()

In [ ]:
scaler = MinMaxScaler()
df_noisy = pd.DataFrame(scaler.fit_transform(df_noisy))
df_noisy.columns = ['x0', 'x1', 'y']

In [ ]:
sigma = 0.02
df_noisy = add_gaussian_noise(
        df=df_noisy.copy(),
        columns=[x for x in df_train.columns],
        mean=0,
        std=sigma
)

In [ ]:
fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection='3d')

ax.scatter(df_noisy['x0'], df_noisy['x1'], df_noisy['y'], marker='o', c=df_noisy['y'], cmap='magma', alpha=0.7)

ax.set_xlabel('x0')
ax.set_ylabel('x1')
ax.set_zlabel('y')

In [ ]:
df_train = pd.DataFrame(scaler.transform(df_train))
df_train.columns = ['x0', 'x1', 'y']
df_train.describe().T

In [ ]:
df_val = pd.DataFrame(scaler.transform(df_val))
df_val.columns = ['x0', 'x1', 'y']
df_val.describe().T

In [ ]:
df_test = pd.DataFrame(scaler.transform(df_test))
df_test.columns = ['x0', 'x1', 'y']
df_test.describe().T

In [ ]:
# Transform the data into a tensor
bio_dataset_train = Dataset(
    input_df=df_noisy[['x0', 'x1']],
    target_df=df_noisy[['y']]
)
# Transform the data into a tensor
bio_dataset_val = Dataset(
    input_df=df_noisy[['x0', 'x1']],
    target_df=df_val[['y']]
)

# Create the dataloaders
batch_size = 64
train_dataloader = DataLoader(bio_dataset_train, batch_size=batch_size, shuffle=True, num_workers=0)
val_dataloader = DataLoader(bio_dataset_val, batch_size=batch_size, shuffle=True, num_workers=0)

In [ ]:
model = GridFullyDenseNN(
    n_layers=2,
    hidden_layers=[
        (2, 1024),
        (1024, 1),
    ],
    dropout_layers=[
        0.0,
        0.0,
    ],
    activation_func_layers=[
        nn.ReLU(),
        nn.Identity(),
    ],
    want_dropout=[
        False,
        False,
    ],
    want_linear=[
        True,
        True,
    ],
    want_activation=[
        True,
        False,
    ],
)
model.to(device)

In [ ]:
lr = 0.001
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=lr)

# Define the trainer
trainer_basic = Trainer(
    model=model,
    train_generator=train_dataloader,
    val_generator=val_dataloader,
    device=device,
    criterion=criterion,
    optimizer=optimizer,
    epoch_scheduler=None,
    batch_scheduler=None,
    patience=15,
    epochs=500,
    checkpoints_path=os.path.join(CHECKPOINT_PATH, f'model_noise_{sigma}_2variables'),
)

In [ ]:
# model, train_losses, val_losses, best_train_loss, best_val_loss = trainer_basic.fit(verbose=True)

In [ ]:
model.load_state_dict(torch.load(os.path.join(CHECKPOINT_PATH, f'model_sin_noise_{sigma}_2variables.pth')))

In [ ]:
y_pred_train = model(torch.tensor(df_noisy[['x0', 'x1']].values.reshape(-1, 2)).float().to(device)).cpu().detach().numpy().reshape(-1)
y_pred_val = model(torch.tensor(df_val[['x0', 'x1']].values.reshape(-1, 2)).float().to(device)).cpu().detach().numpy().reshape(-1)
y_pred_test = model(torch.tensor(df_test[['x0', 'x1']].values.reshape(-1, 2)).float().to(device)).cpu().detach().numpy().reshape(-1)

In [ ]:
mae = mean_absolute_error(df_test['y'].values, y_pred_test)
mse = mean_squared_error(df_test['y'].values, y_pred_test)
rmse = np.sqrt(mse)
mape = np.mean(np.abs((df_test['y'].values - y_pred_test) / df_test['y'].values)) * 100
r_squared = r2_score(df_test['y'].values, y_pred_test)
print("MAE: ", mae)
print("MAPE: ", mape)
print("RMSE: ", rmse)
print("MSE: ", mse)
print("R-squared: ", r_squared)

In [ ]:
fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(df_noisy['x0'], df_noisy['x1'], df_noisy['y'], marker='*', c=df_noisy['y'], cmap='magma', alpha=0.5)
ax.scatter(df_noisy['x0'], df_noisy['x1'], y_pred_train, marker='o', c=y_pred_train, cmap='viridis', alpha=0.5)
ax.legend(['Noisy', 'Predicted'])

ax.set_xlabel('x0')
ax.set_ylabel('x1')
ax.set_zlabel('y')

# Gradients to reduce noise

In [ ]:
df_no_noise = df_noisy.copy()
dlnr = DLNoiseReduction(model=model, criterion=criterion)
dlnr.fit(df_noisy[['x0', 'x1']].values, df_noisy[['y']].values)

In [ ]:
df_no_noise[['x0', 'x1']], df_no_noise['y'] = dlnr.transform(
                                                nrr=0.01,
                                                nr_threshold=0.01,
                                                max_epochs=500,
                                                plot_progress=False
                                            )

In [ ]:
df_no_noise[['x0', 'x1']], df_no_noise['y'] = dlnr.fit_transform(
    X=df_noisy[['x0', 'x1']].values,
    y=df_noisy[['y']].values,
    nrr=0.05,
    nr_threshold=0.01,
    max_epochs=100,
    plot_progress=True,
    path_to_save_imgs='imgs/3D'
)

In [ ]:
dlnr.assert_improvement(
    x_denoised = df_no_noise[['x0', 'x1']].values.reshape(-1, 2),
    y_denoised = df_no_noise[['y']].values.reshape(-1, 1),
    x_orig = df_train[['x0', 'x1']].values.reshape(-1, 2),
    y_orig = df_train[['y']].values.reshape(-1, 1)
)